In [ ]:
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
import matplotlib as mpl
import matplotlib.patheffects as mpe
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import rasterio as rio
import seaborn as sns
from tqdm import tqdm
import xarray as xr
from scipy.stats import mannwhitneyu, ttest_ind, wilcoxon

from project.utils.data import enrich, load_hillshade, sel_subset
from project.utils.metrics import compute_metrics
from project.utils.plotting import ProjectCustomExtent, despine, cbar_halo
from project.utils.paths import get_path

plt.rcParams["figure.dpi"] = 150

# Switzerland LV95 with extended limits to cover 'Hydrological Switzerland'.
lv95 = ProjectCustomExtent(2056, [2474273, 2842000, 1058000, 1325000])

In [ ]:
# Load predictions.
ds = xr.open_zarr(f'{get_path("log_root")}/predictions_revision.zarr').sel(
    time=slice('1993', None), model=['routed_random', 'routed_hard'])

ds['R_pred_mean'] = ds['R_pred'].mean('time').compute()

In [ ]:
# Load HRU polygons.
hrus = gpd.read_file(
    get_path('hru_file'), use_arrow=True).set_index('EZGNR')

rivers = gpd.read_file(f'{get_path("raw_data_root")}/raw/rivernet/', use_arrow=True)

rivers_subset = rivers.loc[rivers.CUM_LEN > 20000]

bg = load_hillshade(
    hillshade_dir=get_path('data_root'),
    dem_path=f'{get_path("raw_data_root")}/raw/merit.zarr',
    hrus=hrus,
    ds=ds,
    use_terrain=False,
    force_recompute=False)

In [ ]:
dss = ds.sel(time=slice('2000', '2009')).sel(model="routed_random")

ds_plain = dss.where((
    (dss.is_lake == 0) &
    (dss.is_regulated_lake == 0) &
    (dss.is_storage_dam == 0) &
    (dss.is_barrage == 0)
).compute(), drop=True)
RS = np.random.RandomState(40)
random_hru_indices = RS.randint(0, len(ds_plain.hru), size=1000)
ds_plain = ds_plain.isel(hru=random_hru_indices).load()

ds_lakes = dss.where((
    (dss.is_lake == 1) &
    (dss.is_regulated_lake == 0) &
    (dss.is_storage_dam == 0) &
    (dss.is_barrage == 0)
).compute(), drop=True).load()

ds_reg_lakes = dss.where((
    (dss.is_lake == 1) &
    (dss.is_regulated_lake == 1)
).compute(), drop=True).load()

ds_barrages = dss.where((
    (dss.is_lake == 0) &
    (dss.is_regulated_lake == 0) &
    (dss.is_storage_dam == 0) &
    (dss.is_barrage == 1)
).compute(), drop=True).load()

ds_floodcontrol = dss.where((
    (dss.is_storage_dam == 1) &
    (dss['is_floodcontrol'] == 1) &
    (dss['is_lake'] == 0)
).compute(), drop=True).load()

ds_hydropower = dss.where((
    (dss.is_storage_dam == 1) &
    (dss.is_hydroelectricity == 1) &
    (dss.is_lake == 0) &
    (dss.is_regulated_lake == 0)
).compute(), drop=True).load()

name_dict = {
    'area': 'DU area (km²)',
    'dam_height': 'Dam height (m)',
    'impoundment_volume': 'Impoundment volume (km³)',
    'slope_mean': 'Slope (°)',
    'upstream_area': 'Upstream area (km²)',
    'water_freq': 'Water freq. (-)',
}

In [ ]:
fig = plt.figure(figsize=(14, 5), constrained_layout=True)
gs = mpl.gridspec.GridSpec(1, 2)

vmin, vmax = 0, 1

for col, model in enumerate(["routed_random", "routed_hard"]):
    map_ax = fig.add_subplot(gs[0, col], projection=lv95)
    bodr = cartopy.feature.NaturalEarthFeature(category='cultural',
        name='admin_0_boundary_lines_land', scale='10m', facecolor='none')
    map_ax.set_global()

    axins = map_ax.inset_axes((0.04, 0.88, 0.25, 0.03))

    r_mean = enrich(
        ds=ds.sel(model=model).R_pred_mean,
        hrus=hrus,
        variables='R_pred_mean'
    )
    im = r_mean.plot(
        column='R_pred_mean',
        cmap='RdPu',
        vmin=vmin,
        vmax=vmax,
        legend=True,
        ec='k',
        lw=0.1,
        alpha=0.8,
        ax=map_ax,
        legend_kwds={'cax': axins, 'orientation': 'horizontal', 'label': 'Retention'}
    )
    cbar_halo(ax=axins, which='x', linewidth=5, color='0.95', labelsize=10, ticksize=8)

    rivers_subset.plot(
        ec='k', fc='none', linewidth=(np.log(rivers_subset.CUM_LEN + 1) / 12 - 0.5),
        alpha=0.8,
        zorder=5,
        ax=map_ax
    )

    lakes_hru = ds.hru.where(ds.is_lake.load(), drop=True)
    r_mean.loc[lakes_hru].plot(
        legend=False,
        ec='none',
        fc='w',
        lw=0.,
        zorder=10,
        ax=map_ax,
    )
    r_mean.loc[lakes_hru].plot(
        column='R_pred_mean',
        cmap='RdPu',
        vmin=vmin,
        vmax=vmax,
        legend=False,
        ec='k',
        lw=0.1,
        alpha=0.8,
        zorder=10,
        ax=map_ax,
    )

    title = "Random routing" if model == "routed_random" else "Hard routing"
    map_ax.set_title(title)

fig.savefig('./figures/retention.png', dpi=300, bbox_inches='tight')

In [ ]:
fig = plt.figure(figsize=(14, 5), constrained_layout=True)

gs = mpl.gridspec.GridSpec(3, 4, width_ratios=[1, 1, 0.7, 0.7])

# Map
# ---------------------------------------------------------------

map_ax = fig.add_subplot(gs[:, :2], projection=lv95)

bodr = cartopy.feature.NaturalEarthFeature(category='cultural', 
    name='admin_0_boundary_lines_land', scale='10m', facecolor='none')

map_ax.set_global()

vmin, vmax = 0, 1

# bg.plot.imshow(ax=map_ax, interpolation='bilinear', alpha=0.8, zorder=0)
# map_ax.add_feature(bodr, linestyle='--', edgecolor='k', alpha=1, zorder=10, lw=0.7)

# axins = map_ax.inset_axes((0.05, 0.95, 0.25, 0.02))
axins = map_ax.inset_axes((0.04, 0.88, 0.25, 0.03))

r_mean = enrich(
    ds=ds.R_pred_mean.sel(model="routed_random"),
    hrus=hrus,
    variables='R_pred_mean'
)
im = r_mean.plot(
    column='R_pred_mean',
    cmap='RdPu',
    vmin=vmin,
    vmax=vmax,
    legend=True,
    ec='k',
    lw=0.1,
    alpha=0.8,
    ax=map_ax,
    legend_kwds={'cax': axins, 'orientation': 'horizontal', 'label': 'Retention'}
)

cbar_halo(ax=axins, which='x', linewidth=5, color='0.95', labelsize=10, ticksize=8)

rivers_subset.plot(
    ec='k', fc='none', linewidth=(np.log(rivers_subset.CUM_LEN + 1) / 12 - 0.5),
    alpha=0.8,
    zorder=5,
    ax=map_ax)

r_mean.loc[ds.hru.where(ds.is_lake.load(), drop=True)].plot(
    legend=False,
    ec='none',
    fc='w',
    lw=0.,
    zorder=10,
    ax=map_ax,
)

r_mean.loc[ds.hru.where(ds.is_lake.load(), drop=True)].plot(
    column='R_pred_mean',
    cmap='RdPu',
    vmin=vmin,
    vmax=vmax,
    legend=False,
    ec='k',
    lw=0.1,
    alpha=0.8,
    zorder=10,
    ax=map_ax,
)

map_ax.set_title('')

# Retention vs Storage
# ---------------------------------------------------------------

axes = []

for r in range(3):
    r_axes = []
    for rec_c, c in enumerate(range(2)):
        if len(axes) == 0:
            ax = fig.add_subplot(gs[r, 2 + c])
        else:
            ax = fig.add_subplot(gs[r, 2 + c], sharey=axes[0][0])
        despine(ax)
        r_axes.append(ax)
    axes.append(r_axes)

axes_flat = [ax for row in axes for ax in row]

i = 0
for ds_type, cvar, name in zip(
    [ds_plain, ds_lakes, ds_reg_lakes, ds_barrages, ds_floodcontrol, ds_hydropower],
    ['water_freq', 'area', 'area', 'slope_mean', 'dam_height', 'dam_height'],
    ['Riverine unmanaged', 'Unregulated Lakes', 'Regulated Lakes', 'Barrages', 'Flood Control', 'Hydropower']):

    r = i // 2
    c = i % 2

    ax = axes_flat[i]

    cvar_nice = name_dict[cvar]

    cmap = 'cool'
    norm = mpl.colors.LogNorm()

    im = ax.scatter(
        ds_type.S_pred_m3 * 60 * 60 * 24,
        ds_type.R_pred, c=ds_type[cvar].expand_dims(time=ds_type.time),
        s=0.5,
        ec='none',
        cmap=cmap,
        norm=norm
    )
    ax.set_xscale('log')

    if i == 3:
        ax.set_xlim(10**4.6, None)

    axins = ax.inset_axes((0.92, 0.1, 0.02, 0.6))
    cbar = fig.colorbar(im, cax=axins, orientation='vertical')
    cbar.set_label(label=cvar_nice, size=8)
    cbar.ax.tick_params(which='both', labelsize=8)
    axins.yaxis.set_ticks_position('left')

    alphan = ['a', 'b', 'c', 'd', 'e', 'f'][i]
    if i == 0:
        num = f''
    else:
        num = f' ($n$={len(ds_type.hru)})'

    ax.text(0.01, 1, f'{alphan}) ' + name + num, ha='left', va='top', transform=ax.transAxes, fontsize=9,
        path_effects=[mpe.withStroke(linewidth=3, foreground="w")])

    if c == 0:
        ax.set_ylabel('Retention (-)')
    else:
        ax.set_ylabel('')

    if r == 2:
        ax.set_xlabel('Storage (m³)')
    else:
        ax.set_xlabel('')

    i += 1

    ax.set_ylim(-0.05, 1.2)

fig.savefig('./figures/retention.png', dpi=300, bbox_inches='tight')

In [ ]:
fig = plt.figure(figsize=(14, 5), constrained_layout=True)

gs = mpl.gridspec.GridSpec(3, 4, width_ratios=[1, 1, 0.7, 0.7])

# Map
# ---------------------------------------------------------------

map_ax = fig.add_subplot(gs[:, :2], projection=lv95)

bodr = cartopy.feature.NaturalEarthFeature(category='cultural', 
    name='admin_0_boundary_lines_land', scale='10m', facecolor='none')

map_ax.set_global()

vmin, vmax = 0, 1

# bg.plot.imshow(ax=map_ax, interpolation='bilinear', alpha=0.8, zorder=0)
# map_ax.add_feature(bodr, linestyle='--', edgecolor='k', alpha=1, zorder=10, lw=0.7)

# axins = map_ax.inset_axes((0.05, 0.95, 0.25, 0.02))
axins = map_ax.inset_axes((0.04, 0.88, 0.25, 0.03))

r_mean = enrich(
    ds=ds.R_pred_mean,
    hrus=hrus,
    variables='R_pred_mean'
)
im = r_mean.plot(
    column='R_pred_mean',
    cmap='RdPu',
    vmin=vmin,
    vmax=vmax,
    legend=True,
    ec='k',
    lw=0.1,
    alpha=0.8,
    ax=map_ax,
    legend_kwds={'cax': axins, 'orientation': 'horizontal', 'label': 'Retention'}
)

cbar_halo(ax=axins, which='x', linewidth=5, color='0.95', labelsize=10, ticksize=8)

rivers_subset.plot(
    ec='k', fc='none', linewidth=(np.log(rivers_subset.CUM_LEN + 1) / 12 - 0.5),
    alpha=0.8,
    zorder=5,
    ax=map_ax)

r_mean.loc[ds.hru.where(ds.is_lake.load(), drop=True)].plot(
    legend=False,
    ec='none',
    fc='w',
    lw=0.,
    zorder=10,
    ax=map_ax,
)

r_mean.loc[ds.hru.where(ds.is_lake.load(), drop=True)].plot(
    column='R_pred_mean',
    cmap='RdPu',
    vmin=vmin,
    vmax=vmax,
    legend=False,
    ec='k',
    lw=0.1,
    alpha=0.8,
    zorder=10,
    ax=map_ax,
)

map_ax.set_title('')

# Retention vs Storage
# ---------------------------------------------------------------

axes = []

for r in range(3):
    r_axes = []
    for rec_c, c in enumerate(range(2)):
        if len(axes) == 0:
            ax = fig.add_subplot(gs[r, 2 + c])
        else:
            ax = fig.add_subplot(gs[r, 2 + c], sharey=axes[0][0])
        despine(ax)
        r_axes.append(ax)
    axes.append(r_axes)

axes_flat = [ax for row in axes for ax in row]

i = 0
for ds_type, cvar, name in zip(
    [ds_plain, ds_lakes, ds_reg_lakes, ds_barrages, ds_floodcontrol, ds_hydropower],
    ['water_freq', 'area', 'area', 'slope_mean', 'dam_height', 'dam_height'],
    ['Riverine unmanaged', 'Unregulated Lakes', 'Regulated Lakes', 'Barrages', 'Flood Control', 'Hydropower']):

    r = i // 2
    c = i % 2

    ax = axes_flat[i]

    cvar_nice = name_dict[cvar]

    cmap = 'cool'
    norm = mpl.colors.LogNorm()

    im = ax.scatter(
        ds_type.S_pred_m3 * 60 * 60 * 24,
        ds_type.R_pred, c=ds_type[cvar].expand_dims(time=ds_type.time),
        s=0.5,
        ec='none',
        cmap=cmap,
        norm=norm
    )
    ax.set_xscale('log')

    if i == 3:
        ax.set_xlim(10**4.6, None)

    axins = ax.inset_axes((0.92, 0.1, 0.02, 0.6))
    cbar = fig.colorbar(im, cax=axins, orientation='vertical')
    cbar.set_label(label=cvar_nice, size=8)
    cbar.ax.tick_params(which='both', labelsize=8)
    axins.yaxis.set_ticks_position('left')

    alphan = ['a', 'b', 'c', 'd', 'e', 'f'][i]
    if i == 0:
        num = f''
    else:
        num = f' ($n$={len(ds_type.hru)})'

    ax.text(0.01, 1, f'{alphan}) ' + name + num, ha='left', va='top', transform=ax.transAxes, fontsize=9,
        path_effects=[mpe.withStroke(linewidth=3, foreground="w")])

    if c == 0:
        ax.set_ylabel('Retention (-)')
    else:
        ax.set_ylabel('')

    if r == 2:
        ax.set_xlabel('Storage (m³)')
    else:
        ax.set_xlabel('')

    i += 1

    ax.set_ylim(-0.05, 1.2)

fig.savefig('./figures/retention.png', dpi=300, bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5), subplot_kw={'projection': lv95}, gridspec_kw={'wspace': 0.02})

bodr = cartopy.feature.NaturalEarthFeature(category='cultural', 
    name='admin_0_boundary_lines_land', scale='10m', facecolor='none')

ax.set_global()

vmin, vmax = 0, 1

bg.plot.imshow(ax=ax, interpolation='bilinear', alpha=0.8, zorder=0)
ax.add_feature(bodr, linestyle='--', edgecolor='k', alpha=1, zorder=10, lw=0.7)

axins = ax.inset_axes((0.05, 0.95, 0.25, 0.02))
im = enrich(
    ds=ds.R_pred_mean.sel(model='routedbackup_random'),
    hrus=hrus,
    variables='R_pred_mean'
).plot(
    column='R_pred_mean',
    cmap='RdPu',
    vmin=vmin,
    vmax=vmax,
    legend=True,
    ec='k',
    lw=0.1,
    alpha=0.6,
    ax=ax,
    legend_kwds={'cax': axins, 'orientation': 'horizontal', 'label': 'Retention'}
)

rivers_subset.plot(
    ec='k', fc='none', linewidth=(np.log(rivers_subset.CUM_LEN + 1) / 12 - 0.5),
    alpha=0.5,
    zorder=5,
    ax=ax)
ax.set_title('Retention')


In [ ]:
ds = xr.open_zarr(f'{get_path("log_root")}/predictions_revision.zarr').sel(
    time=slice('1993', None), model='routed_hard')

In [ ]:
ds.sel(hru=155625).R_pred.plot()

In [ ]:
# Load predictions.
ds = xr.open_zarr(f'{get_path("log_root")}/routed/random/xval/fold_000/preds.zarr/').sel(
    time=slice('1993', None))

ds['R_pred_mean'] = ds['R_pred'].mean('time').compute()

In [ ]:
fig = plt.figure(figsize=(14, 5), constrained_layout=True)

gs = mpl.gridspec.GridSpec(3, 4, width_ratios=[1, 1, 0.7, 0.7])

# Map
# ---------------------------------------------------------------

map_ax = fig.add_subplot(gs[:, :2], projection=lv95)

bodr = cartopy.feature.NaturalEarthFeature(category='cultural', 
    name='admin_0_boundary_lines_land', scale='10m', facecolor='none')

map_ax.set_global()

vmin, vmax = 0, 1

# bg.plot.imshow(ax=map_ax, interpolation='bilinear', alpha=0.8, zorder=0)
# map_ax.add_feature(bodr, linestyle='--', edgecolor='k', alpha=1, zorder=10, lw=0.7)

# axins = map_ax.inset_axes((0.05, 0.95, 0.25, 0.02))
axins = map_ax.inset_axes((0.04, 0.88, 0.25, 0.03))

r_mean = enrich(
    ds=ds.R_pred_mean,
    hrus=hrus,
    variables='R_pred_mean'
)
im = r_mean.plot(
    column='R_pred_mean',
    cmap='RdPu',
    vmin=vmin,
    vmax=vmax,
    legend=True,
    ec='k',
    lw=0.1,
    alpha=0.8,
    ax=map_ax,
    legend_kwds={'cax': axins, 'orientation': 'horizontal', 'label': 'Retention'}
)

cbar_halo(ax=axins, which='x', linewidth=5, color='0.95', labelsize=10, ticksize=8)

rivers_subset.plot(
    ec='k', fc='none', linewidth=(np.log(rivers_subset.CUM_LEN + 1) / 12 - 0.5),
    alpha=0.8,
    zorder=5,
    ax=map_ax)

r_mean.loc[ds.hru.where(dss.is_lake.load(), drop=True)].plot(
    legend=False,
    ec='none',
    fc='w',
    lw=0.,
    zorder=10,
    ax=map_ax,
)

r_mean.loc[ds.hru.where(dss.is_lake.load(), drop=True)].plot(
    column='R_pred_mean',
    cmap='RdPu',
    vmin=vmin,
    vmax=vmax,
    legend=False,
    ec='k',
    lw=0.1,
    alpha=0.8,
    zorder=10,
    ax=map_ax,
)

map_ax.set_title('')

# Retention vs Storage
# ---------------------------------------------------------------

axes = []

for r in range(3):
    r_axes = []
    for rec_c, c in enumerate(range(2)):
        if len(axes) == 0:
            ax = fig.add_subplot(gs[r, 2 + c])
        else:
            ax = fig.add_subplot(gs[r, 2 + c], sharey=axes[0][0])
        despine(ax)
        r_axes.append(ax)
    axes.append(r_axes)

axes_flat = [ax for row in axes for ax in row]

i = 0
for ds_type, cvar, name in zip(
    [ds_plain, ds_lakes, ds_reg_lakes, ds_barrages, ds_floodcontrol, ds_hydropower],
    ['water_freq', 'area', 'area', 'slope_mean', 'dam_height', 'dam_height'],
    ['Riverine unmanaged', 'Unregulated Lakes', 'Regulated Lakes', 'Barrages', 'Flood Control', 'Hydropower']):

    r = i // 2
    c = i % 2

    ax = axes_flat[i]

    cvar_nice = name_dict[cvar]

    cmap = 'cool'
    norm = mpl.colors.LogNorm()

    im = ax.scatter(
        ds_type.S_pred_m3 * 60 * 60 * 24,
        ds_type.R_pred, c=ds_type[cvar].expand_dims(time=ds_type.time),
        s=0.5,
        ec='none',
        cmap=cmap,
        norm=norm
    )
    ax.set_xscale('log')

    if i == 3:
        ax.set_xlim(10**4.6, None)

    axins = ax.inset_axes((0.92, 0.1, 0.02, 0.6))
    cbar = fig.colorbar(im, cax=axins, orientation='vertical')
    cbar.set_label(label=cvar_nice, size=8)
    cbar.ax.tick_params(which='both', labelsize=8)
    axins.yaxis.set_ticks_position('left')

    alphan = ['a', 'b', 'c', 'd', 'e', 'f'][i]
    if i == 0:
        num = f''
    else:
        num = f' ($n$={len(ds_type.hru)})'

    ax.text(0.01, 1, f'{alphan}) ' + name + num, ha='left', va='top', transform=ax.transAxes, fontsize=9,
        path_effects=[mpe.withStroke(linewidth=3, foreground="w")])

    if c == 0:
        ax.set_ylabel('Retention (-)')
    else:
        ax.set_ylabel('')

    if r == 2:
        ax.set_xlabel('Storage (m³)')
    else:
        ax.set_xlabel('')

    i += 1

    ax.set_ylim(-0.05, 1.2)

fig.savefig('./figures/retention.png', dpi=300, bbox_inches='tight')